# Concurrent Execution with Fleche

This notebook shows how to use `@fleche`-decorated functions with Python's three executor types.

`BoundWrapper.bind(func)` is the recommended pattern for sharing cache state across workers: it captures the active cache and metadata at bind time and restores them on every subsequent call — even in subprocesses — without requiring a hand-written wrapper function.

| Executor | Required storage | Recommended pattern |
|---|---|---|
| `ThreadPoolExecutor` | Any (Memory or file) | `BoundWrapper.bind(func)` |
| `ProcessPoolExecutor` | **Persistent** (file / SQL) | `BoundWrapper.bind(func)` |
| `SingleNodeExecutor` (executorlib) | **Persistent** (file / SQL) | `BoundWrapper.bind(func)` |

In [ ]:
import time
import tempfile
import os
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

from fleche import fleche, cache, BoundWrapper
from fleche.caches import Cache
from fleche.storage.memory import ValueMemory, CallMemory
from fleche.storage.pickle_file import ValuePickleFile, CallPickleFile

## 1. ThreadPoolExecutor

`BoundWrapper.bind(func)` captures the active cache at bind time and restores it on every call — including inside worker threads. Use it directly inside a `with cache(...):` block to ensure all submitted tasks write to the same store:

In [ ]:
@fleche
def add(x, y):
    return x + y

In [ ]:
my_cache = Cache(ValueMemory({}), CallMemory({}))

with cache(my_cache):
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = [pool.submit(BoundWrapper.bind(add), x, x + 1) for x in range(4)]
        results = [f.result() for f in futures]

print("Results:", results)                          # [1, 3, 5, 7]
print("In my_cache:", my_cache.contains(add.digest(0, 1)))  # True

### Passing a bound function around

Because the bound callable is a regular Python object, you can bind it once and pass it to other modules, helper functions, or executor pools without re-specifying the cache. The captured state travels with the callable:

In [ ]:
my_cache = Cache(ValueMemory({}), CallMemory({}))

with cache(my_cache):
    bound_add = BoundWrapper.bind(add)   # bind once, reuse anywhere

# bound_add carries my_cache — the cache context manager is no longer needed at the call site
with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(bound_add, x, x + 1) for x in range(4)]
    results = [f.result() for f in futures]

print("Results:", results)                          # [1, 3, 5, 7]
print("In my_cache:", my_cache.contains(add.digest(0, 1)))  # True

---
## 2. ProcessPoolExecutor

Worker processes run in separate Python interpreters — an in-memory cache set in the parent is **not visible** inside workers. Use a **file-** or **SQL-backed** backend so that results written by workers are visible to the parent process.

In [ ]:
@fleche
def expensive(x):
    return x ** 3

with tempfile.TemporaryDirectory() as tmpdir:
    shared_cache = Cache(
        ValuePickleFile.with_pickle(root=os.path.join(tmpdir, "values")),
        CallPickleFile.with_pickle(root=os.path.join(tmpdir, "calls")),
    )

    with cache(shared_cache):
        with ProcessPoolExecutor(max_workers=2) as pool:
            results = list(pool.map(BoundWrapper.bind(expensive), range(5)))

    print("Results:", results)                                      # [0, 1, 8, 27, 64]
    print("Cached:", shared_cache.contains(expensive.digest(3)))   # True

---
## 3. executorlib.SingleNodeExecutor

[executorlib](https://github.com/pyiron/executorlib) is a third-party library for submitting tasks to HPC schedulers and local process pools. Its `SingleNodeExecutor` uses *process-based* workers with the same isolation semantics as `ProcessPoolExecutor`.

> **Note:** `executorlib` is an optional dependency. Install it with `pip install executorlib`.

Use a **file-backed** cache with `BoundWrapper.bind` so that results written by workers are visible to the parent process:

In [ ]:
from executorlib import SingleNodeExecutor

@fleche
def cube_sne(x):
    return x ** 3

with tempfile.TemporaryDirectory() as tmpdir:
    shared_cache = Cache(
        ValuePickleFile.with_pickle(root=os.path.join(tmpdir, "values")),
        CallPickleFile.with_pickle(root=os.path.join(tmpdir, "calls")),
    )

    with cache(shared_cache):
        with SingleNodeExecutor() as executor:
            futures = [executor.submit(BoundWrapper.bind(cube_sne), x) for x in range(5)]
            results = [f.result() for f in futures]

    print("Results:", results)                                          # [0, 1, 8, 27, 64]
    print("In shared_cache:", shared_cache.contains(cube_sne.digest(3)))  # True

### Summary

```
ThreadPoolExecutor
  with cache(...): with ThreadPoolExecutor() as pool:
    pool.submit(BoundWrapper.bind(func), ...)     ✅ workers share the same cache

ProcessPoolExecutor / executorlib.SingleNodeExecutor
  In-memory cache                               ❌ not visible in worker processes
  File/SQL cache + BoundWrapper.bind(func)      ✅ worker results visible to parent
```

**Storage reference**

| Backend | Cross-process sharing |
|---|---|
| `Memory` | ❌ Ephemeral — process-local only |
| `PickleFile` | ✅ Persistent — shared via filesystem |
| `Sql` | ✅ Persistent — shared via database |
| `BagOfHolding` | ✅ Persistent — shared via HDF5 file |